# Mcardiac: infer a transition prior

Estimate a transition map with UOT when a prior file is unavailable.


## Settings


In [ ]:
from pathlib import Path
import yaml

DATASET = 'Mcardiac'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
TRANSITION_PRIOR = experiment_path(CONFIG["transition_prior_path"]) if "transition_prior_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


## Estimate transitions


In [ ]:
import sys
sys.path.insert(0, str(EXPERIMENT_DIR))
import scanpy as sc
from infer_transition_map import infer_transition_map

adata = sc.read_h5ad(SCANVI_ADATA)
stats, edges = infer_transition_map(
    adata, source="E9.5h", target="E11.5h", label_key="mapped_celltype",
    latent_key=CONFIG["latent_key"], permutations=500, seed=2026, device="cpu",
)
display(stats)
display(edges)


## Save the transition map


In [ ]:
if edges.empty or not (edges["src_layer"] != edges["tgt_layer"]).any():
    raise ValueError("No non-self transition was discovered; inspect edge statistics.")
OUTPUT = EXPERIMENT_DIR / "artifacts/transition_uot"
OUTPUT.mkdir(parents=True, exist_ok=False)
stats.to_csv(OUTPUT / "edge_statistics.csv", index=False)
edges.to_csv(OUTPUT / "discovered_transition.csv", index=False)
print("Set transition_prior_path in config.yaml to artifacts/transition_uot/discovered_transition.csv")
